### 1.2 Формулировка

**Входы**

- Задание и результаты первого знакомства с данными

**Выходы**

- Что предсказываем и по каким данным
- Требования к данным
- Полный набор метрик качества и правила выбора модели и порога

**Что делаем**

Собираем постановку задачи: что подаём модели, что хотим получить и как поймём, насколько хорошо она работает

#### Для кого и когда строим прогноз

Предположим, что заёмщик уже подал заявку на кредит. Кредитор оценил её, назначил грейд `loan_grade` и предложил ставку `loan_int_rate`\
Модель нужна инвестору, который решает, финансировать ли кредит с уже назначенными грейдом и ставкой. В этом сценарии оба признака доступны до прогноза; какие из них использовать, решим после анализа. Сам сценарий — наше допущение: автор не уточняет, как и на каких данных кредитор назначал грейд и ставку

Целевая переменная — `loan_status`: 0 — без дефолта, 1 — дефолт. Будем предсказывать её по остальным столбцам\
Решаем задачу бинарной классификации. Модель выдаёт оценку риска, которую по выбранному порогу переводим в класс 0 или 1

#### Что подаём модели и что получаем

| Что определяем | В нашем случае |
| --- | --- |
| Одна запись | Заявка на кредит |
| Вход | Признаки заявки, отобранные после анализа из 11 исходных полей |
| Результат | Оценка риска, а после применения порога — класс 0 или 1 |

Алгоритм пока не выбираем: сначала разберём данные

#### Какие данные нужны

Для обучения нужен известный исход кредита — `loan_status`, это целевая переменная\
Для новых заявок нужны выбранные для модели поля в исходных единицах измерения

Используем только то, что известно в момент прогноза. В нашем сценарии грейд и ставка уже есть, а фактического исхода кредита ещё нет\
Пропуски, повторы и подозрительные значения проверим при EDA. Период, за который автор определял дефолт, не указан: модель будет предсказывать метку в том виде, в котором она дана в источнике

#### Критерии оценивания

Положительный класс — дефолт: `loan_status = 1`. По AP выберем модель, по F2 — порог. Ниже зададим метрики, которые рассчитаем при итоговой оценке

| Обозначение | Число кредитов |
| --- | --- |
| **TP** | Верно предсказан дефолт |
| **FP** | Предсказан дефолт, фактически его нет |
| **FN** | Дефолт не предсказан, фактически он произошёл |
| **TN** | Верно предсказано отсутствие дефолта |

##### Метрики по предсказанным классам

| Метрика | Формула | Что показывает |
| --- | --- | --- |
| Accuracy | $\frac{TP+TN}{TP+FP+FN+TN}$ | Доля верных ответов |
| Precision (точность) | $\frac{TP}{TP+FP}$ | Доля настоящих дефолтов среди предсказанных |
| Recall (полнота) | $\frac{TP}{TP+FN}$ | Доля обнаруженных дефолтов среди всех фактических |
| F1 | $\frac{2TP}{2TP+FP+FN}$ | Объединяет precision и recall с равными весами |
| F2 | $\frac{5TP}{5TP+FP+4FN}$ | Объединяет precision и recall с большим весом recall |
| Balanced accuracy | $\frac{1}{2}\left(\frac{TP}{TP+FN}+\frac{TN}{TN+FP}\right)$ | Средняя полнота по двум классам |

Для всех метрик этой группы больше — лучше

##### Метрики по оценкам риска

**Average Precision (AP)** — среднее precision, взвешенное приростами recall при снижении порога

$$AP=\sum_i (R_i-R_{i-1})P_i,\qquad P_i=\frac{TP_i}{TP_i+FP_i},\quad R_i=\frac{TP_i}{TP_i+FN_i}$$

Здесь $i$ — номер порога при последовательном снижении, $P_i$ и $R_i$ — precision и recall при этом пороге, $R_0=0$. AP принимает значения от 0 до 1; больше — лучше

**ROC-AUC** — площадь под ROC-кривой. При разных порогах она сопоставляет полноту (TPR) с долей ложных тревог среди кредитов без дефолта (FPR):

$$TPR=\frac{TP}{TP+FN},\qquad FPR=\frac{FP}{FP+TN}$$

Для бинарной классификации ROC-AUC также можно рассчитать по всем парам «дефолт — без дефолта»:

$$\mathrm{ROC\!\text{-}\!AUC}=\frac{N_{>}+0{,}5N_{=}}{n_+n_-}$$

Здесь $n_+$ — число кредитов с дефолтом, $n_-$ — без дефолта; $N_{>}$ — число пар, в которых оценка риска дефолтного кредита выше, $N_{=}$ — число пар с равными оценками. Правильно упорядоченная пара даёт 1, равные оценки — 0,5, обратный порядок — 0\
Значения от 0 до 1; больше — лучше. ROC-AUC = 1 означает, что все дефолты получили оценки выше всех кредитов без дефолта; 0,5 соответствует случайному ранжированию. Как и AP, эта метрика не привязана к одному выбранному порогу

**Brier score** — средняя квадратичная ошибка прогноза вероятности

$$\mathrm{Brier}=\frac{1}{n}\sum_{j=1}^{n}(s_j-y_j)^2$$

Здесь $s_j$ — прогноз вероятности дефолта, $y_j\in\{0,1\}$ — фактический исход, $n$ — число кредитов. Значения от 0 до 1; меньше — лучше

##### Правила выбора и итоговой оценки

Основной критерий сравнения моделей — AP\
Пропустить будущий дефолт хуже, чем ошибочно отнести надёжную заявку к рискованным. Поэтому при выборе порога отдаём приоритет полноте и используем F2, которая сильнее учитывает recall

| Что оцениваем или выбираем | Правило |
| --- | --- |
| Модель | Наибольшая AP на validation |
| Порог | Наибольшая F2 на validation для выбранной модели; при равенстве — больший порог |
| Итоговое качество | На test рассчитываем AP, ROC-AUC, Brier, Accuracy, Precision, Recall, F1, F2 и Balanced accuracy, показываем TP, FP, FN и TN |